In [ ]:
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn

In [ ]:
data = Path("../datasets/logatec")
results_path = Path("artifacts/02-Changed_dataset_to_logatec-results")
random_seed = 42
n_splits = 5
subsets = ["spring", "winter"]

In [ ]:
def load_raw_data(path: Path) -> pd.DataFrame:
    with open(path, mode="r") as fp:
        data = json.load(fp)

    df = []

    for position, measurements in data.items():
        digits = re.findall(r"\d+", position)
        location = tuple(int(i) for i in digits)

        # Winter dataset has measurements only in the middle (3rd) row.
        if len(location) == 1:
            location = (3, *location)

        assert len(location) == 2, f"location identifier is not length 2: {location}"

        pos_x, pos_y = location

        for device_id, samples in measurements.items():
            device_id = int(device_id)
            for sample in samples:
                timestamp, value = sample["timestamp"], sample["rss"]

                item = {"pos_x": pos_x, "pos_y": pos_y, "node": device_id, "timestamp": timestamp, "value": value}
                df.append(item)

    df = pd.DataFrame(df)
    df.timestamp = pd.to_datetime(df.timestamp, unit="s", origin="unix").astype("datetime64[s]")
    df = df.astype({"pos_x": "uint8", "pos_y": "uint8", "value": "int8", "node": "uint8"})

    return df


In [ ]:
# Praparation
# We assume the dataset has been downloaded and unzipped manually

import json
import re

df = [load_raw_data(data / f"{subsets[0]}_data.json"), load_raw_data(data / f"{subsets[1]}_data.json")]

for idx, subset in enumerate(subsets):
    df[idx] = (
        df[idx]
        .groupby(["pos_x", "pos_y", "node", "timestamp"], as_index=False)["value"]
        .mean()
        .pivot(index=["timestamp", "pos_x", "pos_y"], columns="node", values="value")
        .reset_index()
    )
    df[idx].columns = [
        f"node{column}" if isinstance(column, (int, np.integer)) else str(column)
        for column in df[idx].columns
    ]
    df[idx] = df[idx].fillna(-180).drop(columns="timestamp")

In [ ]:
#Feature generation
for idx, subset in enumerate(subsets):
    # Convert discrete values to meters
    df[idx].pos_x = (df[idx].pos_x - 1) * 1.2  # meters
    df[idx].pos_y = (df[idx].pos_y - 1) * 1.2  # meters
    
    df[idx] = df[idx].rename(columns={"pos_x": "target_x", "pos_y": "target_y"})

# Find target column(s)
targets = ["target_x", "target_y"]

# X are features, y are target(s)
features, targets = [df[0].drop(targets, axis=1), df[1].drop(targets, axis=1)], [df[0][targets], df[1][targets]]

In [ ]:
#Split generation
from sklearn import model_selection

groups = None

cv = model_selection.KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_seed,
    )

indices = indices = [[] for _ in range(len(subsets))]

for idx, subset in enumerate(subsets):
    for train_indices, test_indices in cv.split(features[idx], targets[idx], groups):
                indices[idx].append((train_indices, test_indices))


In [ ]:
class PredefinedSplit(model_selection.BaseCrossValidator):
    """Simple cross-validator for predefined train-test splits."""

    def __init__(self, indices_pairs: list[tuple[np.ndarray, np.ndarray]]):
        self.idx_pairs = indices_pairs

    def get_n_splits(self, X=None, y=None, groups=None):
        """Return the number of splitting iterations in the cross-validator"""
        return len(self.idx_pairs)

    def split(self, X, y=None, groups=None):
        """Generate indices to split data into training and test set."""
        for train_idx, test_idx in self.idx_pairs:
            yield train_idx, test_idx

In [ ]:
#Train&Evaluate
from sklearn.ensemble import RandomForestRegressor 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import make_scorer, root_mean_squared_error

for idx, subset in enumerate(subsets):
    cv = PredefinedSplit(indices[idx])
    
    estimators = [
        RandomForestRegressor(random_state=42),
        KNeighborsRegressor()
    ]
    params=[
        {
            "n_estimators": [10, 50, 100, 250, 400], 
            "max_depth": [5, 10, 30, 50, 150, 200, None]
        },
        {
            "n_neighbors": [3, 5, 10], 
            "weights": ["uniform", "distance"], 
            "p": [1, 2], 
            "leaf_size": [10, 15, 30], 
            "metric": ["minkowski", "euclidean"] 
        }
    ]
    
    for index in range(len(estimators)):
        gs = model_selection.GridSearchCV(
            estimator = estimators[index],
            param_grid = params[index],
            n_jobs = -1,
            error_score = "raise",
            refit = True,
            scoring = make_scorer(root_mean_squared_error, greater_is_better=False),
            cv = cv,
        )
        
        gs.fit(features[idx], targets[idx])
        
        results_df = pd.DataFrame(gs.cv_results_)
        
        # Select key columns to display
        cols_to_show = [
            'params',
            'mean_test_score',
            'std_test_score',
            'rank_test_score',
            'mean_fit_time',
            'mean_score_time',
        ]
        
        # Print as a table
        print(results_df[cols_to_show].to_string(index=False))
        Path(results_path).mkdir(parents=True, exist_ok=True)
        joblib.dump(gs.best_estimator_, results_path / f"Model_{estimators[index].__class__.__name__}-KFoldSplit-{subset}Subset.pkl") 
        joblib.dump(results_df, results_path / f"Results_{estimators[index].__class__.__name__}-KFoldSplit-{subset}Subset.pkl")